In [ ]:
"""now , anet with memory ,(from langgraph.checkpoint.memory import MemorySaver

# Add memory to save conversation history
memory = MemorySaver()

app = workflow.compile(checkpointer=memory)

# Use with thread_id for conversation continuity
config = {"configurable": {"thread_id": "user-123"}}
result = app.invoke(state, config))...  explian deep anlyze it and how this iwll help ful in real world scnecs , how where it was using , in real world how this will lead , give me examples with codes , how to use it ..
In 2025, Memory in LangGraph is a game-changer for Agentic AI because it transforms a "forgetful" chatbot into a persistent "digital employee". 
By using MemorySaver, your agent gains short-term memory (session context) which allows it to recall previous turns without you having to manually re-send the entire chat history every time. 
1. Deep Analysis of the Code
The code you provided implements Checkpointing, which is LangGraph's core mechanism for persistence. 
2. Real-World Use Case: Personal Travel Assistant
In a real-world scenario, an agent without memory would forget a user's destination halfway through the booking process. An agent with memory keeps track of preferences across multiple interactions.
Real-World Code Example
python
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START

# 1. Setup the Graph with memory
builder = StateGraph(MessagesState)
memory = MemorySaver() # Note: Use PostgresSaver or RedisSaver in production

# (Assume nodes for booking are added here)
app = builder.compile(checkpointer=memory)

# 2. User 1 - First Interaction
config_1 = {"configurable": {"thread_id": "user_123"}}
app.invoke({"messages": [("user", "I want to fly to Paris.")]}, config_1)

# 3. User 1 - Follow-up (Agent REMEMBERS "Paris")
# The agent will know 'there' refers to Paris
response = app.invoke({"messages": [("user", "What is the weather like there?")]}, config_1)
print(response["messages"][-1].content) 

# 4. User 2 - Separate Conversation
config_2 = {"configurable": {"thread_id": "user_456"}}
# This agent has no idea about Paris because it's a different thread_id
Use code with caution.

3. How Memory Leads the Way in 2025
Memory-enabled agents are currently being used in high-impact industries to reduce manual work and improve customer loyalty. 
Multi-Step Customer Support: Agents can handle "Tier-1" support by remembering a user's previous troubleshooting steps. They don't ask the user to "restart the router" if the user already said they did it five minutes ago.
Human-in-the-Loop Approval: Because the state is saved, an agent can "pause" and wait for a human to approve a task (like a $10,000 bank transfer). Once approved, the agent "resumes" exactly from that point.
Contextual Financial Advisors: Banks use these agents to remember a customer's specific goals (e.g., "saving for a house") throughout a month-long planning conversation.
Resilience & Fault Tolerance: If a server crashes during a complex 10-step AI workflow, the agent doesn't restart from scratch. It reads its last "checkpoint" and continues from step 9. 
Pro Tip for 2025: For production apps, replace MemorySaver (which clears when the app restarts) with a persistent database checkpointer like PostgresSaver or RedisSaver to ensure your agent's memory survives server reboots. 




undefined
undefined
undefined
17 sites
Adding memory to your Chatbot using LangGraph | Part 3
11 Jun 2025 — Let's get started. * Note: This blog continues from where we left off in our previous blog, where we constructed a chat...

Codersarts

Understanding Memory Management in LangGraph
6 Nov 2025 — What is Memory in LangGraph? Memory in LangGraph refers to the system's ability to maintain conversation state across mu...

Towards AI

Using LangGraph Memory for Persistent Chat Conversations - Medium
30 Mar 2025 — For now, I'll focus on the theoretical concepts, but I may cover the implementation details once I've completed that pa...

Medium

Show all
MemorySaver(): This is an in-memory checkpointer. It takes "snapshots" of the graph's state (messages, variables, and tool outputs) at every step.
workflow.compile(checkpointer=memory): This attaches the memory to the graph logic, enabling the system to automatically save and load state.
thread_id: This is a unique identifier (like a session ID). When you use the same thread_id, the agent "resumes" from where it left off in that specific conversation room.
Automatic Retrieval: When you call .invoke() with a thread_id, LangGraph checks its database (or RAM), finds the last saved state for that thread, and loads it into the model's brain before processing the new message. """

In [20]:
# Install dependencies
!pip install langchain-google-genai langgraph langchain-core python-dotenv


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# Imports
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage

In [25]:
# Load API credentials from .env file
from dotenv import load_dotenv
load_dotenv()

# API key will be loaded from .env file (GOOGLE_API_KEY)
print("✅ Environment variables loaded")
print(f"GOOGLE_API_KEY loaded: {'Yes' if os.environ.get('GOOGLE_API_KEY') else 'No'}")

✅ Environment variables loaded
GOOGLE_API_KEY loaded: Yes


In [ ]:
# Initialize LLM using Google Gemini API
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.environ.get('GOOGLE_API_KEY')
)
print("✅ Gemini LLM initialized")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Gemini LLM initialized


In [27]:
# Define the chatbot node
def chatbot(state: MessagesState):
    """Process messages and generate response using LLM"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Add Memory - This is the key part!
memory = MemorySaver()

# Compile with checkpointer
app = builder.compile(checkpointer=memory)
print("✅ Agent with Memory created!")

✅ Agent with Memory created!


## 🧪 Test 1: Agent Remembers Context

User 1 says "My name is John" → Agent remembers
User 1 asks "What's my name?" → Agent recalls "John"

In [28]:
# User 1 - First message (introduce name)
config_user1 = {"configurable": {"thread_id": "user_john_123"}}

response1 = app.invoke(
    {"messages": [HumanMessage(content="Hi! My name is John and I'm planning a trip to Paris.")]},
    config_user1
)

print("🤖 Agent Response:")
print(response1["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
# User 1 - Follow-up (Agent should REMEMBER name and Paris)
response2 = app.invoke(
    {"messages": [HumanMessage(content="What's my name and where am I going?")]},
    config_user1  # Same thread_id!
)

print("🤖 Agent Response (Should remember John and Paris):")
print(response2["messages"][-1].content)

## 🧪 Test 2: Different Users Have Separate Memory

User 2 (different thread_id) should NOT know about John or Paris

In [ ]:
# User 2 - Different thread_id (New conversation, no memory of John)
config_user2 = {"configurable": {"thread_id": "user_mary_456"}}

response3 = app.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config_user2  # Different thread_id!
)

print("🤖 Agent Response (Should NOT know any name):")
print(response3["messages"][-1].content)

## 🧪 Test 3: Continue Previous Conversation

Go back to User 1's thread and continue the conversation

In [ ]:
# Back to User 1 - Continue conversation (Still remembers everything!)
response4 = app.invoke(
    {"messages": [HumanMessage(content="Can you recommend 3 things to do there?")]},
    config_user1  # Same thread_id as before!
)

print("🤖 Agent Response (Should know 'there' means Paris):")
print(response4["messages"][-1].content)

In [ ]:
# View full conversation history for User 1
print("📜 Full Conversation History for User 1:")
print("=" * 50)
for msg in response4["messages"]:
    role = "👤 User" if isinstance(msg, HumanMessage) else "🤖 Agent"
    print(f"{role}: {msg.content[:200]}...")
    print("-" * 50)

## 🌍 Real-World Use Cases

| Use Case | How Memory Helps |
|----------|------------------|
| **Customer Support** | Remember previous troubleshooting steps |
| **Travel Assistant** | Remember destination, preferences across sessions |
| **Financial Advisor** | Track customer's investment goals over weeks |
| **Healthcare** | Remember patient symptoms from previous visits |

## ⚠️ Production Note
- `MemorySaver()` stores in RAM - data lost on restart
- For production, use `PostgresSaver` or `RedisSaver` for persistent storage